In [ ]:
import os, sys

# In Colab: clone and install from GitHub
# Locally: add repo root to sys.path so opto is importable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %cd /content
    !rm -rf Trace  # clean slate
    !git clone https://github.com/AgentOpt/OpenTrace.git Trace
    %cd Trace
    !git checkout experimental
    !pip install -e .
    !pip install cvxpy matplotlib pandas
    _repo_root = os.getcwd()  # /content/Trace after %cd
else:
    # Local: ensure repo root is on sys.path
    _nb_dir = os.path.dirname(os.path.abspath("__file__"))
    _repo_root = os.path.abspath(os.path.join(_nb_dir, "..", ".."))
    if _repo_root not in sys.path:
        sys.path.insert(0, _repo_root)
    import opto
    print(f"Using local opto from: {os.path.dirname(opto.__file__)}")

print(f"Repo root: {_repo_root}")

# Verify cvxpy is available (required for SixHumpCamel SOS certificate)
try:
    import cvxpy
    print(f"cvxpy {cvxpy.__version__} available")
except ImportError:
    raise ImportError("cvxpy is required: pip install cvxpy")

# T6 M2 — BeamsearchAlgorithm & PrioritySearch Multi-Objective

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/OpenTrace/blob/experimental/examples/notebooks/multiobjective_trainers.ipynb)

**Milestone 2 Deliverable** — Multi-objective support in BeamsearchAlgorithm and PrioritySearch

This notebook demonstrates M2 using Allen's SixHumpCamel convex function environment:
1. **BasicSearch** (M1 baseline): scalar, weighted, and Pareto modes
2. **BeamsearchAlgorithm** (M2): weighted mode with `select()` vector path
3. **PrioritySearch** (M2): weighted and Pareto modes with `ParetoHeapMemory`

All cells run in **StubLLM mode** (no API keys required). Uses `DummyLLM` for deterministic training.

---

## How to Validate This Milestone

After running all cells, confirm:
- [ ] BasicSearch trains with scalar, weighted, and Pareto `objective_config` (M1 baseline)
- [ ] BeamsearchAlgorithm trains with weighted `objective_config` and populates `_last_selected_score_dicts`
- [ ] PrioritySearch trains with weighted and Pareto `objective_config`
- [ ] PrioritySearch memory contains candidates with `score_dict` in rollouts
- [ ] Score progression graph shows per-mode validation curves
- [ ] Comparison scatter shows `base_loss` vs `reg_loss` for all algorithm+mode combos
- [ ] Summary table aggregates results across all runs
- [ ] Backward compatibility: `objective_config=None` trains identically to pre-M2 behavior

In [ ]:
import re
import copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Tuple, Dict

from opto import trace
from opto.trainer.guide import Guide
from opto.trainer.objectives import ObjectiveConfig
from opto.utils.llm import DummyLLM
from opto.optimizers import OptoPrimeV2
from opto.trainer.examples.basic_algorithms import BasicSearchAlgorithm
from opto.trainer.examples.beamsearch_algorithm import BeamsearchAlgorithm
from opto.trainer.algorithms.priority_search import PrioritySearch

# Single-item dataset used by all algorithms (SixHumpCamel ignores inputs/infos)
DATASET = dict(inputs=[None], infos=[None])

print("=" * 70)
print("T6 M2 — BeamsearchAlgorithm & PrioritySearch Multi-Objective")
print("=" * 70)

---
## Part A: SixHumpCamel Convex Function — All Three Algorithms

Uses Allen's `SixHumpCamel` loss landscape with L2-squared regularization.
Two objectives (both to **minimize**):
- `base_loss`: the six-hump camel function value
- `reg_loss`: the L2-squared regularization term

Known global optima near `[0.0898, -0.7126]` and `[-0.0898, 0.7126]`.

### Setup

In [ ]:
# Import SixHumpCamel from Allen's example file
_examples_dir = os.path.join(_repo_root, 'examples')
if _examples_dir not in sys.path:
    sys.path.insert(0, _examples_dir)
from multi_objective_convex_fn import SixHumpCamel


# --- RewardGuide (defined here with correct copy import) ---
class RewardGuide(Guide):
    """Multi-objective guide for convex function environments.

    Both get_feedback() and get_score_dict() evaluate on a deepcopy of
    the environment so that candidate scoring never advances the real env.
    The SixHumpCamel loss computation is stateless (only depends on x),
    so deepcopy is safe and avoids burning through the env's horizon
    during multi-candidate validation.
    """

    def __init__(self, env):
        self.env = env

    def _score_on_copy(self, response):
        """Evaluate response on a deepcopy — env state stays frozen."""
        env_copy = copy.deepcopy(self.env)
        obs, reward, done, info = env_copy.step(str(response))
        return obs, reward, done, info

    def get_feedback(self, query, response, reference=None, **kwargs) -> Tuple[float, str]:
        obs, reward, done, info = self._score_on_copy(response)
        feedback = ((obs + "\n\n") if obs else "") + info.get("feedback", "")
        return float(reward), feedback

    def get_score_dict(self, query, response, reference=None, **kwargs) -> Dict[str, float]:
        obs, reward, done, info = self._score_on_copy(response)
        base_loss = info.get("base_loss")
        reg_loss = info.get("reg_loss")
        if base_loss is None or reg_loss is None:
            base_loss = float("inf")
            reg_loss = float("inf")
        return {"base_loss": float(base_loss), "reg_loss": float(reg_loss)}


# --- Agent: wraps a trace node that holds the x = [x1, x2] string ---
@trace.model
class ConvexAgent:
    def __init__(self, initial_value):
        self.param = trace.node(
            initial_value, trainable=True,
            description="Input x into the hidden function to minimize y. Format: x = [x1, x2]"
        )

    def forward(self, x):
        return self.param


# --- DummyLLM callable: proposes x = [float, float] values ---
class ConvexLLMCallable:
    """Returns cycling proposals spanning the SixHumpCamel landscape."""

    PROPOSALS = [
        "x = [0.09, -0.71]",    # very close to optimum 1
        "x = [-0.09, 0.71]",    # very close to optimum 2
        "x = [0.1, -0.7]",      # near optimum 1
        "x = [-0.1, 0.7]",      # near optimum 2
        "x = [0.5, -0.3]",      # moderate region
        "x = [-0.5, 0.3]",      # moderate symmetric
        "x = [0.2, -0.5]",      # exploring
        "x = [-0.3, 0.6]",      # exploring
        "x = [1.0, -1.0]",      # far from optima (high reg)
        "x = [0.0, 0.0]",       # origin (zero loss)
    ]

    def __init__(self):
        self.idx = 0

    def __call__(self, messages, **kwargs):
        problem = messages[1]["content"]
        name = re.findall(r'<variable name="\s*(.*?)" type=.*>', problem)
        name = name[0] if name else "unknown"
        value = self.PROPOSALS[self.idx % len(self.PROPOSALS)]
        self.idx += 1
        return (
            f"<reasoning> Exploring the loss landscape. </reasoning>\n"
            f"<variable>\n"
            f"<name> {name} </name>\n"
            f"<value> {value} </value>\n"
            f"</variable>"
        )


# --- Post-training evaluation: get actual losses from the final parameter ---
def evaluate_final_losses(param_value):
    """Evaluate a parameter string on a fresh SixHumpCamel env.

    Returns dict with base_loss, reg_loss, total_loss (all actual values).
    """
    env = SixHumpCamel(horizon=200, norm_coef=1.0, seed=42)
    env.reset(seed=42)
    x, stop = env.text_extract(str(param_value))
    if x is None:
        return {"base_loss": float("nan"), "reg_loss": float("nan"), "total_loss": float("nan")}
    base, reg, total = env._eval_losses(x)
    return {"base_loss": float(base), "reg_loss": float(reg), "total_loss": float(total)}


# --- Factory: create fresh env + agent + optimizer + guide per run ---

def make_basicsearch_run():
    env = SixHumpCamel(horizon=200, norm_coef=1.0, seed=42)
    env.reset(seed=42)
    guide = RewardGuide(env)
    agent = ConvexAgent("x = [0.0, 0.0]")
    llm = DummyLLM(ConvexLLMCallable())
    optimizer = OptoPrimeV2(agent.parameters(), llm=llm)
    algo = BasicSearchAlgorithm(agent, optimizer)
    return algo, guide, agent


def make_beamsearch_run():
    env = SixHumpCamel(horizon=200, norm_coef=1.0, seed=42)
    env.reset(seed=42)
    guide = RewardGuide(env)
    agent = ConvexAgent("x = [0.0, 0.0]")
    llm = DummyLLM(ConvexLLMCallable())
    optimizer = OptoPrimeV2(agent.parameters(), llm=llm)
    algo = BeamsearchAlgorithm(agent, optimizer)
    return algo, guide, agent


def make_priority_search_run():
    env = SixHumpCamel(horizon=200, norm_coef=1.0, seed=42)
    env.reset(seed=42)
    guide = RewardGuide(env)
    agent = ConvexAgent("x = [0.0, 0.0]")
    llm = DummyLLM(ConvexLLMCallable())
    optimizer = OptoPrimeV2(agent.parameters(), llm=llm)
    algo = PrioritySearch(agent, optimizer)
    return algo, guide, agent


# Objective configs
CONFIG_WEIGHTED = ObjectiveConfig(
    mode="weighted",
    weights={"base_loss": 1.0, "reg_loss": 1.0},
    minimize=frozenset({"base_loss", "reg_loss"}),
    seed=0,
)

CONFIG_PARETO = ObjectiveConfig(
    mode="pareto",
    weights={"base_loss": 0.7, "reg_loss": 0.3},
    minimize=frozenset({"base_loss", "reg_loss"}),
    tie_break="weighted",
    seed=42,
)

# Results collector
results = {}
print("Setup complete. SixHumpCamel environment + DummyLLM ready.")
print(f"DummyLLM has {len(ConvexLLMCallable.PROPOSALS)} diverse proposals.")

In [ ]:
# =====================================================================
# BasicSearch: scalar, weighted, pareto (M1 baseline for comparison)
# num_epochs=5 gives 5 training steps (1-item dataset, batch_size=1)
# =====================================================================
BASIC_KWARGS = dict(
    train_dataset=DATASET,
    num_proposals=4,
    num_epochs=5,
    batch_size=1,
    num_threads=1,
)

# --- Scalar ---
print("=" * 60)
print("BasicSearch: SCALAR mode (5 epochs)")
print("=" * 60)
algo_bs_scalar, guide_bs_scalar, agent_bs_scalar = make_basicsearch_run()
scores_bs_scalar, test_bs_scalar = algo_bs_scalar.train(
    guide=guide_bs_scalar, objective_config=None, **BASIC_KWARGS
)
eval_scalar = evaluate_final_losses(agent_bs_scalar.param.data)
results['BasicSearch/scalar'] = {
    'val_scores': scores_bs_scalar,
    'final_score': test_bs_scalar,
    'eval_losses': eval_scalar,
    'final_param': str(agent_bs_scalar.param.data),
}
print(f"\nFinal param: {agent_bs_scalar.param.data}")
print(f"Validation scores ({len(scores_bs_scalar)} steps): {scores_bs_scalar}")
print(f"Evaluated losses: {eval_scalar}")

# --- Weighted ---
print("\n" + "=" * 60)
print("BasicSearch: WEIGHTED mode (5 epochs)")
print("=" * 60)
algo_bs_weighted, guide_bs_weighted, agent_bs_weighted = make_basicsearch_run()
scores_bs_weighted, test_bs_weighted = algo_bs_weighted.train(
    guide=guide_bs_weighted, objective_config=CONFIG_WEIGHTED, **BASIC_KWARGS
)
eval_weighted = evaluate_final_losses(agent_bs_weighted.param.data)
results['BasicSearch/weighted'] = {
    'val_scores': scores_bs_weighted,
    'final_score': test_bs_weighted,
    'eval_losses': eval_weighted,
    'final_param': str(agent_bs_weighted.param.data),
}
print(f"\nFinal param: {agent_bs_weighted.param.data}")
print(f"Validation scores ({len(scores_bs_weighted)} steps): {scores_bs_weighted}")
print(f"Evaluated losses: {eval_weighted}")

# --- Pareto ---
print("\n" + "=" * 60)
print("BasicSearch: PARETO mode (5 epochs)")
print("=" * 60)
algo_bs_pareto, guide_bs_pareto, agent_bs_pareto = make_basicsearch_run()
scores_bs_pareto, test_bs_pareto = algo_bs_pareto.train(
    guide=guide_bs_pareto, objective_config=CONFIG_PARETO, **BASIC_KWARGS
)
eval_pareto = evaluate_final_losses(agent_bs_pareto.param.data)
results['BasicSearch/pareto'] = {
    'val_scores': scores_bs_pareto,
    'final_score': test_bs_pareto,
    'eval_losses': eval_pareto,
    'final_param': str(agent_bs_pareto.param.data),
}
print(f"\nFinal param: {agent_bs_pareto.param.data}")
print(f"Validation scores ({len(scores_bs_pareto)} steps): {scores_bs_pareto}")
print(f"Evaluated losses: {eval_pareto}")

In [ ]:
# =====================================================================
# Graph 1: Score Progression — BasicSearch validation scores across modes
# With 5 epochs, each mode produces 5 data points showing optimization
# =====================================================================
fig, ax = plt.subplots(1, 1, figsize=(9, 5))

for label, marker, color in [
    ('BasicSearch/scalar', 'o-', '#1f77b4'),
    ('BasicSearch/weighted', 's-', '#ff7f0e'),
    ('BasicSearch/pareto', '^-', '#2ca02c'),
]:
    data = results.get(label, {})
    val_scores = data.get('val_scores', [])
    if val_scores:
        steps = list(range(len(val_scores)))
        ax.plot(steps, val_scores, marker, color=color, label=label,
                linewidth=2, markersize=7)

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Validation Score (reward)', fontsize=12)
ax.set_title('BasicSearch Score Progression — SixHumpCamel (5 epochs)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Score progression across 5 training epochs.")
print("Higher reward = lower total loss (reward = -total_loss).")
print("Different objective modes may select different proposals, producing different curves.")

---
## Part B: BeamsearchAlgorithm (M2)

Tests the `select()` method's vector scoring path:
- `evaluate_vector()` computes per-metric scores for each beam candidate
- `select_top_k()` ranks candidates using the `ObjectiveConfig`
- Per-metric validation scores are logged (e.g., `Validation score/base_loss`)
- `_last_selected_score_dicts` stores the score_dicts from the final selection

In [ ]:
# =====================================================================
# BeamsearchAlgorithm: weighted mode
# max_depth=2 gives 2 beam search levels with validation at each
# =====================================================================
print("=" * 60)
print("Beamsearch: WEIGHTED mode (depth=2, width=3)")
print("=" * 60)

algo_beam_w, guide_beam_w, agent_beam_w = make_beamsearch_run()
metrics_beam_w, final_beam_w = algo_beam_w.train(
    guide=guide_beam_w,
    train_dataset=DATASET,
    objective_config=CONFIG_WEIGHTED,
    beam_width=3,
    num_proposals=3,
    max_depth=2,
    batch_size=1,
    num_threads=1,
)

# Post-training evaluation on fresh env
eval_beam_w = evaluate_final_losses(agent_beam_w.param.data)

# Extract score_dicts from selection
beam_score_dicts = getattr(algo_beam_w, '_last_selected_score_dicts', None)
beam_best_sd = beam_score_dicts[0] if beam_score_dicts else None

results['Beamsearch/weighted'] = {
    'val_scores': metrics_beam_w.get('best_validation_scores', []),
    'final_score': final_beam_w,
    'eval_losses': eval_beam_w,
    'final_param': str(agent_beam_w.param.data),
}

print(f"\nFinal param: {agent_beam_w.param.data}")
print(f"Validation scores by depth: {metrics_beam_w.get('best_validation_scores', [])}")
print(f"_last_selected_score_dicts: {beam_score_dicts}")
print(f"Evaluated losses: {eval_beam_w}")

---
## Part C: PrioritySearch (M2)

Tests the full PrioritySearch multi-objective pipeline:
- `validate()` populates `score_dict` in rollouts via `guide.get_score_dict()`
- `ModuleCandidate.mean_score_dict()` aggregates per-metric means across rollouts
- `compute_exploration_priority()` uses weighted scalarization when `objective_config` is set
- **Weighted mode:** `HeapMemory` with weighted scalar priority
- **Pareto mode:** `ParetoHeapMemory` with Pareto-front-aware `pop()`

In [ ]:
# =====================================================================
# PrioritySearch: weighted mode (more epochs for real exploration)
# =====================================================================
print("=" * 60)
print("PrioritySearch: WEIGHTED mode (2 epochs, 2 batches)")
print("=" * 60)

algo_ps_w, guide_ps_w, agent_ps_w = make_priority_search_run()
algo_ps_w.train(
    guide=guide_ps_w,
    train_dataset=DATASET,
    objective_config=CONFIG_WEIGHTED,
    batch_size=1,
    num_batches=2,
    num_epochs=2,
    num_candidates=3,
    num_proposals=2,
    num_threads=1,
    long_term_memory_size=10,
    memory_update_frequency=0,
    verbose=False,
)

# Post-training evaluation on fresh env
eval_ps_w = evaluate_final_losses(agent_ps_w.param.data)

# Extract best candidate from memory
ps_w_sd = None
ps_w_final = None
if hasattr(algo_ps_w, 'long_term_memory') and algo_ps_w.long_term_memory:
    best_neg, best_cand = min(algo_ps_w.long_term_memory, key=lambda x: x[0])
    ps_w_sd = best_cand.mean_score_dict()
    ps_w_final = float(-best_neg)
    print(f"\nBest candidate priority: {ps_w_final:.4f}")
    print(f"Best candidate mean_score_dict: {ps_w_sd}")
    has_sd = any('score_dict' in r and r['score_dict'] is not None
                 for r in best_cand.rollouts)
    print(f"Rollouts contain score_dict: {has_sd}")
else:
    print("No candidates in long_term_memory")

print(f"Final param: {agent_ps_w.param.data}")
print(f"Evaluated losses: {eval_ps_w}")

results['PrioritySearch/weighted'] = {
    'val_scores': [],
    'final_score': ps_w_final,
    'eval_losses': eval_ps_w,
    'final_param': str(agent_ps_w.param.data),
}

In [ ]:
# =====================================================================
# PrioritySearch: Pareto mode (uses ParetoHeapMemory)
# =====================================================================
print("=" * 60)
print("PrioritySearch: PARETO mode (2 epochs, 2 batches)")
print("=" * 60)

algo_ps_p, guide_ps_p, agent_ps_p = make_priority_search_run()
algo_ps_p.train(
    guide=guide_ps_p,
    train_dataset=DATASET,
    objective_config=CONFIG_PARETO,
    batch_size=1,
    num_batches=2,
    num_epochs=2,
    num_candidates=3,
    num_proposals=2,
    num_threads=1,
    long_term_memory_size=10,
    memory_update_frequency=0,
    verbose=False,
)

# Post-training evaluation on fresh env
eval_ps_p = evaluate_final_losses(agent_ps_p.param.data)

# Extract best candidate from memory
ps_p_sd = None
ps_p_final = None
if hasattr(algo_ps_p, 'long_term_memory') and algo_ps_p.long_term_memory:
    mem_type = type(algo_ps_p.long_term_memory).__name__
    print(f"Memory type: {mem_type}")

    best_neg_p, best_cand_p = min(algo_ps_p.long_term_memory, key=lambda x: x[0])
    ps_p_sd = best_cand_p.mean_score_dict()
    ps_p_final = float(-best_neg_p)
    print(f"\nBest candidate priority: {ps_p_final:.4f}")
    print(f"Best candidate mean_score_dict: {ps_p_sd}")
    has_sd_p = any('score_dict' in r and r['score_dict'] is not None
                   for r in best_cand_p.rollouts)
    print(f"Rollouts contain score_dict: {has_sd_p}")

    print(f"\nAll candidates in memory ({len(algo_ps_p.long_term_memory)}):")
    for neg_p, cand in sorted(algo_ps_p.long_term_memory, key=lambda x: x[0]):
        sd = cand.mean_score_dict()
        print(f"  priority={-neg_p:.4f}, score_dict={sd}")
else:
    print("No candidates in long_term_memory")

print(f"Final param: {agent_ps_p.param.data}")
print(f"Evaluated losses: {eval_ps_p}")

results['PrioritySearch/pareto'] = {
    'val_scores': [],
    'final_score': ps_p_final,
    'eval_losses': eval_ps_p,
    'final_param': str(agent_ps_p.param.data),
}

In [ ]:
# =====================================================================
# Graph 2: Comparison Scatter — base_loss vs reg_loss
# Each point = final parameter evaluated on a fresh SixHumpCamel env
# =====================================================================
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

markers = {
    'BasicSearch': 'o',
    'Beamsearch': 's',
    'PrioritySearch': '^',
}
colors = {
    'scalar': '#1f77b4',
    'weighted': '#ff7f0e',
    'pareto': '#2ca02c',
}

for run_name, run_data in results.items():
    el = run_data.get('eval_losses')
    if el is None or 'base_loss' not in el or 'reg_loss' not in el:
        continue
    # Skip NaN entries (e.g. if text_extract failed)
    if np.isnan(el['base_loss']) or np.isnan(el['reg_loss']):
        continue
    algo_name, mode_name = run_name.split('/')
    ax.scatter(
        el['base_loss'], el['reg_loss'],
        marker=markers.get(algo_name, 'x'),
        color=colors.get(mode_name, 'gray'),
        s=120, edgecolors='black', linewidths=0.8,
        label=run_name, zorder=5,
    )

ax.set_xlabel('base_loss (lower is better)', fontsize=12)
ax.set_ylabel('reg_loss (lower is better)', fontsize=12)
ax.set_title('Multi-Objective Comparison — base_loss vs reg_loss', fontsize=14)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Graph 2: Each point represents the final parameter evaluated on a fresh SixHumpCamel env.")
print("Ideal candidates are in the bottom-left (low base_loss AND low reg_loss).")

In [ ]:
# =====================================================================
# Summary Table
# =====================================================================
rows = []
for run_name, run_data in results.items():
    algo_name, mode_name = run_name.split('/')
    el = run_data.get('eval_losses')
    rows.append({
        'Algorithm': algo_name,
        'Mode': mode_name,
        'Final Scalar Score': f"{run_data.get('final_score', 'N/A')}",
        'base_loss': f"{el['base_loss']:.4f}" if el and 'base_loss' in el and not np.isnan(el['base_loss']) else 'N/A',
        'reg_loss': f"{el['reg_loss']:.4f}" if el and 'reg_loss' in el and not np.isnan(el['reg_loss']) else 'N/A',
        'total_loss': f"{el['total_loss']:.4f}" if el and 'total_loss' in el and not np.isnan(el['total_loss']) else 'N/A',
        'Final Param': run_data.get('final_param', 'N/A'),
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("SUMMARY: Multi-Objective Training Results")
print("=" * 70)
print(df.to_string(index=False))

print("\n" + "=" * 70)
print("M2 NOTEBOOK COMPLETE")
print("=" * 70)
print("""
Deliverables verified:
  Part A (BasicSearch): scalar, weighted, Pareto modes on SixHumpCamel
    - Backward compatible (objective_config=None)
    - Weighted mode populates current_score_dict
    - Pareto mode selects from non-dominated front

  Part B (BeamsearchAlgorithm): weighted mode with vector select()
    - evaluate_vector() computes per-metric scores for beam candidates
    - select_top_k() ranks candidates via ObjectiveConfig
    - _last_selected_score_dicts populated for per-metric logging

  Part C (PrioritySearch): weighted + Pareto modes
    - validate() populates score_dict in rollouts
    - mean_score_dict() aggregates per-metric means
    - compute_exploration_priority() uses weighted scalarization
    - ParetoHeapMemory used in Pareto mode
    - Rollouts contain score_dict entries
""")